# Notebook 07 — Uncertainty and Applicability Domain

**Project:** CMT Path A — leakage-audited multi-ion computed insertion-electrode benchmark
**Purpose:** quantify prediction uncertainty, conformal interval coverage, and applicability-domain distance for the validation benchmark.

This notebook uses outputs from Notebook 02 and Notebook 04. It does **not** do candidate ranking, CDE matching, criticality filtering, sodium case-study triage, or manuscript writing.


In [ ]:
# ============================================================
# Notebook 07: Uncertainty and Applicability Domain
# CMT Path A: Leakage-audited multi-ion computed electrode benchmark
# ============================================================
#
# Purpose:
#   Use Notebook 02 protocol features and Notebook 04 benchmark context to build:
#     1. tree-ensemble predictive uncertainty,
#     2. split-conformal prediction intervals,
#     3. kNN descriptor-space applicability-domain distance,
#     4. in-domain / borderline / out-of-domain flags,
#     5. uncertainty-vs-error diagnostics.
#
# Strict exclusions in this notebook:
#   - No candidate ranking
#   - No sodium case study
#   - No CDE matching
#   - No criticality filtering
#   - No manuscript writing
# ============================================================

from __future__ import annotations

import os
import sys
import re
import json
import math
import time
import platform
import warnings
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# Suppress repeated sklearn/joblib UI spam warnings that can freeze browser output.
warnings.filterwarnings(
    "ignore",
    message=".*sklearn.utils.parallel.delayed.*",
    category=UserWarning,
)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, GroupKFold, ShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# -
# Server-aware CPU settings
# -
SERVER_TOTAL_LOGICAL_CORES = os.cpu_count() or 24
# Your server has 2x12 CPU cores. This leaves headroom for Windows/Jupyter and avoids browser/kernel starvation.
N_JOBS_TREE_MODELS = min(18, max(1, SERVER_TOTAL_LOGICAL_CORES - 6))

# -
# Notebook behavior controls
# -
RANDOM_SEED = 20260704
ALPHA = 0.05  # 95% split-conformal intervals
N_ESTIMATORS_UNCERTAINTY = 400
MIN_TRAIN_ROWS = 100
MIN_TEST_ROWS = 5
MIN_CALIBRATION_ROWS = 100
CALIBRATION_FRACTION = 0.20
KNN_K = 5

# Reviewer-core defaults. Keep these strict and manageable.
PRIMARY_TARGETS = [
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_worst",
]
PROTOCOLS_FOR_UNCERTAINTY = ["P1", "P2", "P3"]
SPLITS_FOR_UNCERTAINTY = [
    "random_split",
    "framework_groupkfold",
    "leave_family_out",
    "leave_chemical_system_out",
    "leave_working_ion_out",
]

# Leave-one-group-out controls for speed and stability.
MAX_LEAVE_CHEMSYS_GROUPS = 75
MAX_LEAVE_FAMILY_GROUPS = None  # None = all family groups with enough records.
RANDOM_SPLIT_REPEATS = 5
GROUPKFOLD_N_SPLITS = 5

# Do not include P0/P4 by default because uncertainty should be reported for clean/decision-support protocols.
INCLUDE_LEAKY_PROTOCOLS_FOR_DIAGNOSTICS = False

# -
# Canonical clean-room input and output namespaces
# -
def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

NB09_DIR = artifact_namespace("02", REPOSITORY_ROOT)
NB10_DIR = artifact_namespace("04", REPOSITORY_ROOT)

BASE_DIR = artifact_namespace("07", REPOSITORY_ROOT)
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"
CHECKPOINT_DIR = runtime_cache_root(REPOSITORY_ROOT) / "notebook_07" / "checkpoints"

for d in [PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "07_event_log.csv", index=False)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

def safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except Exception:
        pass
    return str(x)

log_event("init", "INFO", "Notebook 07 initialized.", {"nb09_dir": str(NB09_DIR), "nb10_dir": str(NB10_DIR)})
print(f"Notebook 02 input: {NB09_DIR}")
print(f"Notebook 04 input: {NB10_DIR}")
print(f"Notebook 07 output: {BASE_DIR}")
print(f"Detected logical cores: {SERVER_TOTAL_LOGICAL_CORES}; tree model n_jobs: {N_JOBS_TREE_MODELS}")


In [ ]:
# ============================================================
# Cell 2: Save environment/config metadata
# ============================================================

software_environment = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": package_version("scikit-learn"),
    "scipy_version": package_version("scipy"),
    "notebook_name": "07_uncertainty_and_applicability_domain.ipynb",
}

benchmark_config = {
    "random_seed": RANDOM_SEED,
    "alpha": ALPHA,
    "interval_confidence": 1 - ALPHA,
    "n_estimators_uncertainty": N_ESTIMATORS_UNCERTAINTY,
    "n_jobs_tree_models": N_JOBS_TREE_MODELS,
    "primary_targets": PRIMARY_TARGETS,
    "protocols_for_uncertainty": PROTOCOLS_FOR_UNCERTAINTY,
    "splits_for_uncertainty": SPLITS_FOR_UNCERTAINTY,
    "random_split_repeats": RANDOM_SPLIT_REPEATS,
    "groupkfold_n_splits": GROUPKFOLD_N_SPLITS,
    "max_leave_chemsys_groups": MAX_LEAVE_CHEMSYS_GROUPS,
    "max_leave_family_groups": MAX_LEAVE_FAMILY_GROUPS,
    "min_train_rows": MIN_TRAIN_ROWS,
    "min_test_rows": MIN_TEST_ROWS,
    "min_calibration_rows": MIN_CALIBRATION_ROWS,
    "calibration_fraction": CALIBRATION_FRACTION,
    "knn_k": KNN_K,
    "no_candidate_ranking": True,
    "no_cde_matching": True,
    "no_criticality_filtering": True,
    "no_manuscript_writing": True,
}

write_json_safe(software_environment, METADATA_DIR / "07_software_environment.json")
write_json_safe(benchmark_config, METADATA_DIR / "07_uncertainty_config.json")

display(pd.DataFrame([software_environment]))
display(pd.DataFrame([benchmark_config]))
save_event_log()


In [ ]:
# ============================================================
# Cell 3: Load Notebook 02 and Notebook 04 outputs
# ============================================================

master_feature_path = NB09_DIR / "processed" / "02_master_feature_table.csv"
mask_path = NB09_DIR / "audit" / "02_target_plausibility_masks.csv"
protocol_lookup_path = NB09_DIR / "processed" / "02_protocol_lookup_for_notebook_10.csv"
protocol_integrity_path = NB09_DIR / "audit" / "02_protocol_integrity_audit.csv"
nb09_decision_path = NB09_DIR / "metadata" / "02_final_decision.json"

nb10_best_path = NB10_DIR / "processed" / "04_best_model_by_target_protocol_split.csv"
nb10_agg_path = NB10_DIR / "processed" / "04_benchmark_aggregate_results.csv"
nb10_decision_path = NB10_DIR / "metadata" / "04_final_decision.json"

required_files = [
    master_feature_path,
    mask_path,
    protocol_lookup_path,
    protocol_integrity_path,
    nb10_best_path,
    nb10_agg_path,
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing_files))

feature_df = pd.read_csv(master_feature_path, low_memory=False)
mask_df = pd.read_csv(mask_path, low_memory=False)
protocol_lookup_df = pd.read_csv(protocol_lookup_path)
protocol_integrity_df = pd.read_csv(protocol_integrity_path)
nb10_best_df = pd.read_csv(nb10_best_path)
nb10_agg_df = pd.read_csv(nb10_agg_path)

if nb09_decision_path.exists():
    nb09_decision = json.load(open(nb09_decision_path, "r", encoding="utf-8"))
else:
    nb09_decision = {}

if nb10_decision_path.exists():
    nb10_decision = json.load(open(nb10_decision_path, "r", encoding="utf-8"))
else:
    nb10_decision = {}

print("Loaded inputs:")
print(f"  feature_df:          {feature_df.shape}")
print(f"  mask_df:             {mask_df.shape}")
print(f"  protocol_lookup_df:  {protocol_lookup_df.shape}")
print(f"  nb10_best_df:        {nb10_best_df.shape}")
print(f"  nb10_agg_df:         {nb10_agg_df.shape}")
print(f"  Notebook 02 decision: {nb09_decision.get('final_decision', 'unknown')}")
print(f"  Notebook 04 decision: {nb10_decision.get('final_decision', 'unknown')}")

# Basic safety checks.
if (protocol_integrity_df["status"] == "FAIL").any():
    raise RuntimeError("Notebook 02 protocol integrity has FAIL rows. Do not run Notebook 07.")

if nb10_decision.get("final_decision") not in {"FULL_GO_TO_NOTEBOOK_11", "unknown", None}:
    log_event(
        "input_decision",
        "WARNING",
        "Notebook 04 final decision was not FULL_GO_TO_NOTEBOOK_11.",
        {"nb10_decision": nb10_decision},
    )

# Merge masks into master feature table.
mask_cols = [c for c in mask_df.columns if c.startswith("mask_") or c in ["record_index", "electrode_uid"]]
feature_df = feature_df.merge(mask_df[mask_cols], on=["record_index", "electrode_uid"], how="left")

# Ensure key target/group columns exist.
required_cols = [
    "record_index", "electrode_uid", "working_ion", "framework_uid",
    "chemical_system_uid", "host_chemsys_no_working_ion", "coarse_family"
] + PRIMARY_TARGETS
missing_cols = [c for c in required_cols if c not in feature_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in master feature table: {missing_cols}")

display(feature_df.head())
save_event_log()


In [ ]:
# ============================================================
# Cell 4: Helper functions for features, splits, metrics, and intervals
# ============================================================

def load_protocol_features(target: str, protocol: str) -> list[str]:
    row = protocol_lookup_df[
        (protocol_lookup_df["target"] == target) &
        (protocol_lookup_df["protocol"] == protocol)
    ]
    if row.empty:
        raise ValueError(f"No protocol lookup row for target={target}, protocol={protocol}")

    csv_path = Path(row.iloc[0]["feature_list_csv"])
    if not csv_path.exists():
        # Support running from different working directory by resolving relative to project root.
        alt = NB09_DIR / "processed" / "protocol_feature_lists" / csv_path.name
        if alt.exists():
            csv_path = alt
        else:
            raise FileNotFoundError(f"Feature list file missing: {csv_path}")

    fdf = pd.read_csv(csv_path)
    if "feature" not in fdf.columns:
        raise ValueError(f"Feature list has no feature column: {csv_path}")

    features = [f for f in fdf["feature"].astype(str).tolist() if f in feature_df.columns]
    return features


def target_direction(target: str) -> str:
    # Higher voltage, capacity, energy desirable; lower volume/stability desirable.
    if target in {"max_delta_volume", "stability_charge", "stability_discharge", "stability_worst"}:
        return "minimize"
    return "maximize"


def target_mask_for(target: str) -> str:
    # Use conservative all-target plausible mask for comparable benchmark.
    if "mask_physics_plausible_all_targets" in feature_df.columns:
        return "mask_physics_plausible_all_targets"
    if "mask_physics_plausible_primary" in feature_df.columns:
        return "mask_physics_plausible_primary"
    return "mask_raw_all_records"


def prepare_xy(target: str, protocol: str):
    features = load_protocol_features(target, protocol)
    mask_col = target_mask_for(target)

    df = feature_df.copy()
    df = df[df[mask_col].astype(bool)]
    df = df[df[target].notna()]

    # Keep numeric features with at least one valid value.
    valid_features = []
    for f in features:
        vals = pd.to_numeric(df[f], errors="coerce")
        if vals.notna().sum() > 0:
            df[f] = vals
            valid_features.append(f)

    if not valid_features:
        raise ValueError(f"No valid features for target={target}, protocol={protocol}")

    y = pd.to_numeric(df[target], errors="coerce")
    ok = y.notna()
    df = df.loc[ok].reset_index(drop=True)
    y = y.loc[ok].to_numpy(dtype=float)
    X = df[valid_features].copy()

    return df, X, y, valid_features, mask_col


def make_splits(df: pd.DataFrame, split_name: str, seed: int = RANDOM_SEED):
    n = len(df)
    indices = np.arange(n)
    splits = []

    if split_name == "random_split":
        rs = ShuffleSplit(n_splits=RANDOM_SPLIT_REPEATS, test_size=0.20, random_state=seed)
        for fold_id, (train_idx, test_idx) in enumerate(rs.split(indices)):
            splits.append((fold_id, "random_" + str(fold_id), train_idx, test_idx))
        return splits

    if split_name == "framework_groupkfold":
        groups = df["framework_uid"].astype(str).fillna("unknown_framework").to_numpy()
        n_groups = len(np.unique(groups))
        n_splits = min(GROUPKFOLD_N_SPLITS, n_groups)
        if n_splits < 2:
            return []
        gkf = GroupKFold(n_splits=n_splits)
        for fold_id, (train_idx, test_idx) in enumerate(gkf.split(indices, groups=groups)):
            heldout = "|".join(sorted(pd.Series(groups[test_idx]).unique().tolist())[:25])
            splits.append((fold_id, heldout, train_idx, test_idx))
        return splits

    if split_name == "leave_family_out":
        groups = df["coarse_family"].astype(str).replace("", "unknown_family")
        counts = groups.value_counts()
        eligible = counts[counts >= MIN_TEST_ROWS]
        if MAX_LEAVE_FAMILY_GROUPS is not None:
            eligible = eligible.head(MAX_LEAVE_FAMILY_GROUPS)
        for fold_id, heldout_group in enumerate(eligible.index.tolist()):
            test_idx = np.where(groups.to_numpy() == heldout_group)[0]
            train_idx = np.setdiff1d(indices, test_idx)
            splits.append((fold_id, heldout_group, train_idx, test_idx))
        return splits

    if split_name == "leave_chemical_system_out":
        groups = df["host_chemsys_no_working_ion"].astype(str).replace("", "unknown_chemsys")
        counts = groups.value_counts()
        eligible = counts[counts >= MIN_TEST_ROWS].head(MAX_LEAVE_CHEMSYS_GROUPS)
        for fold_id, heldout_group in enumerate(eligible.index.tolist()):
            test_idx = np.where(groups.to_numpy() == heldout_group)[0]
            train_idx = np.setdiff1d(indices, test_idx)
            splits.append((fold_id, heldout_group, train_idx, test_idx))
        return splits

    if split_name == "leave_working_ion_out":
        groups = df["working_ion"].astype(str).replace("", "unknown_ion")
        counts = groups.value_counts()
        eligible = counts[counts >= MIN_TEST_ROWS]
        for fold_id, heldout_group in enumerate([g for g in ["Li", "Na", "K"] if g in eligible.index]):
            test_idx = np.where(groups.to_numpy() == heldout_group)[0]
            train_idx = np.setdiff1d(indices, test_idx)
            splits.append((fold_id, heldout_group, train_idx, test_idx))
        return splits

    raise ValueError(f"Unknown split_name: {split_name}")


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def spearman_safe(y_true, y_pred):
    if len(y_true) < 3:
        return np.nan
    if np.nanstd(y_true) == 0 or np.nanstd(y_pred) == 0:
        return np.nan
    if SCIPY_AVAILABLE:
        try:
            return float(spearmanr(y_true, y_pred, nan_policy="omit").correlation)
        except Exception:
            return np.nan
    try:
        return float(pd.Series(y_true).corr(pd.Series(y_pred), method="spearman"))
    except Exception:
        return np.nan


def conformal_quantile(abs_residuals, alpha=ALPHA):
    r = np.asarray(abs_residuals, dtype=float)
    r = r[np.isfinite(r)]
    n = len(r)
    if n == 0:
        return np.nan
    # Split conformal finite-sample quantile index.
    q_level = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(r, q_level, method="higher"))


def percentile_against_reference(values, reference):
    ref = np.asarray(reference, dtype=float)
    ref = ref[np.isfinite(ref)]
    vals = np.asarray(values, dtype=float)
    if len(ref) == 0:
        return np.full_like(vals, np.nan, dtype=float)
    # Percentile of each value against reference distribution.
    return np.array([100.0 * np.mean(ref <= v) for v in vals], dtype=float)


def ad_flag_from_percentile(p):
    if not np.isfinite(p):
        return "unknown"
    if p <= 90.0:
        return "in_domain"
    if p <= 97.0:
        return "borderline"
    return "out_of_domain"

print("Helper functions ready.")


In [ ]:
# ============================================================
# Cell 5: Build execution plan
# ============================================================

protocols = list(PROTOCOLS_FOR_UNCERTAINTY)
if INCLUDE_LEAKY_PROTOCOLS_FOR_DIAGNOSTICS:
    protocols = ["P0"] + protocols + ["P4"]

config_rows = []
for target in PRIMARY_TARGETS:
    for protocol in protocols:
        try:
            feats = load_protocol_features(target, protocol)
            n_feats = len(feats)
        except Exception as exc:
            log_event("config_plan", "ERROR", "Could not load feature list.", {"target": target, "protocol": protocol, "error": str(exc)})
            n_feats = 0

        for split_name in SPLITS_FOR_UNCERTAINTY:
            config_key = f"{target}_{protocol}_{split_name}"
            config_rows.append({
                "config_key": config_key,
                "target": target,
                "protocol": protocol,
                "split_name": split_name,
                "n_protocol_features_available": n_feats,
            })

config_plan_df = pd.DataFrame(config_rows)
config_plan_df.to_csv(AUDIT_DIR / "07_uncertainty_config_plan.csv", index=False)

display(config_plan_df)
print(f"Total uncertainty configs planned: {len(config_plan_df)}")
save_event_log()


In [ ]:
# ============================================================
# Cell 6: Load checkpoints if available
# ============================================================

PRED_CHECKPOINT = CHECKPOINT_DIR / "07_uncertainty_predictions_checkpoint.csv"
FOLD_CHECKPOINT = CHECKPOINT_DIR / "07_uncertainty_fold_metrics_checkpoint.csv"
CONFIG_CHECKPOINT = CHECKPOINT_DIR / "07_config_execution_checkpoint.csv"

if CONFIG_CHECKPOINT.exists():
    completed_config_df = pd.read_csv(CONFIG_CHECKPOINT)
    completed_ok = set(completed_config_df.loc[completed_config_df["status"] == "OK", "config_key"].astype(str))
else:
    completed_config_df = pd.DataFrame()
    completed_ok = set()

print(f"Completed OK configs found in checkpoint: {len(completed_ok)}")
if completed_ok:
    display(pd.DataFrame({"completed_config_key": sorted(completed_ok)}).head(20))


In [ ]:
# ============================================================
# Cell 7: Core uncertainty fold runner
# ============================================================

def fit_predict_uncertainty_one_fold(
    df: pd.DataFrame,
    X: pd.DataFrame,
    y: np.ndarray,
    features: list[str],
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    target: str,
    protocol: str,
    split_name: str,
    fold_id: int,
    heldout_group: str,
):
    t0 = time.time()

    if len(train_idx) < MIN_TRAIN_ROWS or len(test_idx) < MIN_TEST_ROWS:
        raise ValueError(f"Insufficient train/test rows: train={len(train_idx)}, test={len(test_idx)}")

    X_train_all = X.iloc[train_idx]
    y_train_all = y[train_idx]
    X_test = X.iloc[test_idx]
    y_test = y[test_idx]

    # Split training fold into model-fit and calibration subsets.
    if len(train_idx) >= (MIN_TRAIN_ROWS + MIN_CALIBRATION_ROWS):
        fit_local_idx, calib_local_idx = train_test_split(
            np.arange(len(train_idx)),
            test_size=CALIBRATION_FRACTION,
            random_state=RANDOM_SEED + 1000 + fold_id,
        )
        X_fit = X_train_all.iloc[fit_local_idx]
        y_fit = y_train_all[fit_local_idx]
        X_calib = X_train_all.iloc[calib_local_idx]
        y_calib = y_train_all[calib_local_idx]
        calibration_mode = "split_conformal_holdout_from_training_fold"
    else:
        X_fit = X_train_all
        y_fit = y_train_all
        X_calib = X_train_all
        y_calib = y_train_all
        calibration_mode = "in_sample_fallback_small_training_fold"

    # Imputation.
    imputer = SimpleImputer(strategy="median")
    X_fit_imp = imputer.fit_transform(X_fit)
    X_calib_imp = imputer.transform(X_calib)
    X_test_imp = imputer.transform(X_test)

    # Model.
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS_UNCERTAINTY,
        random_state=RANDOM_SEED + fold_id,
        n_jobs=N_JOBS_TREE_MODELS,
        bootstrap=True,
        max_features="sqrt",
        min_samples_leaf=1,
    )
    model.fit(X_fit_imp, y_fit)

    y_calib_pred = model.predict(X_calib_imp)
    y_test_pred = model.predict(X_test_imp)

    # Per-tree prediction distribution.
    tree_preds = np.vstack([est.predict(X_test_imp) for est in model.estimators_]).T
    tree_mean = tree_preds.mean(axis=1)
    tree_std = tree_preds.std(axis=1, ddof=1)
    tree_q025 = np.quantile(tree_preds, 0.025, axis=1)
    tree_q975 = np.quantile(tree_preds, 0.975, axis=1)

    # Conformal residual interval.
    calib_abs_resid = np.abs(y_calib - y_calib_pred)
    qhat = conformal_quantile(calib_abs_resid, alpha=ALPHA)
    conformal_lower = y_test_pred - qhat
    conformal_upper = y_test_pred + qhat
    conformal_width = conformal_upper - conformal_lower
    covered = (y_test >= conformal_lower) & (y_test <= conformal_upper)

    # Applicability-domain distance.
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit_imp)
    X_test_scaled = scaler.transform(X_test_imp)

    k = min(KNN_K, len(X_fit_scaled))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean", n_jobs=1)
    nn.fit(X_fit_scaled)
    test_dists, _ = nn.kneighbors(X_test_scaled)
    test_knn_mean_distance = test_dists.mean(axis=1)
    test_knn_min_distance = test_dists.min(axis=1)

    # Reference train-domain distances: distance to nearest neighbor excluding self.
    if len(X_fit_scaled) > 1:
        train_k = min(k + 1, len(X_fit_scaled))
        nn_train = NearestNeighbors(n_neighbors=train_k, metric="euclidean", n_jobs=1)
        nn_train.fit(X_fit_scaled)
        train_dists, _ = nn_train.kneighbors(X_fit_scaled)
        # If self is first neighbor, use columns 1:; otherwise still use mean excluding first.
        train_ref = train_dists[:, 1:].mean(axis=1) if train_dists.shape[1] > 1 else train_dists[:, 0]
    else:
        train_ref = np.array([np.nan])

    domain_pct = percentile_against_reference(test_knn_mean_distance, train_ref)
    ad_flags = np.array([ad_flag_from_percentile(p) for p in domain_pct], dtype=object)

    abs_error = np.abs(y_test - y_test_pred)
    sq_error = (y_test - y_test_pred) ** 2

    pred_rows = pd.DataFrame({
        "target": target,
        "protocol": protocol,
        "split_name": split_name,
        "fold_id": fold_id,
        "heldout_group": heldout_group,
        "model_name": "ExtraTrees_uncertainty",
        "record_index": df.iloc[test_idx]["record_index"].to_numpy(),
        "electrode_uid": df.iloc[test_idx]["electrode_uid"].to_numpy(),
        "working_ion": df.iloc[test_idx]["working_ion"].to_numpy(),
        "coarse_family": df.iloc[test_idx]["coarse_family"].to_numpy(),
        "chemical_system_uid": df.iloc[test_idx]["chemical_system_uid"].to_numpy(),
        "host_chemsys_no_working_ion": df.iloc[test_idx]["host_chemsys_no_working_ion"].to_numpy(),
        "true_value": y_test,
        "predicted_value": y_test_pred,
        "absolute_error": abs_error,
        "squared_error": sq_error,
        "tree_pred_mean": tree_mean,
        "tree_pred_std": tree_std,
        "tree_q025": tree_q025,
        "tree_q975": tree_q975,
        "conformal_qhat": qhat,
        "conformal_lower": conformal_lower,
        "conformal_upper": conformal_upper,
        "conformal_width": conformal_width,
        "conformal_covered": covered.astype(int),
        "knn_mean_distance": test_knn_mean_distance,
        "knn_min_distance": test_knn_min_distance,
        "domain_distance_percentile": domain_pct,
        "ad_flag": ad_flags,
        "n_train": len(train_idx),
        "n_fit": len(X_fit),
        "n_calibration": len(X_calib),
        "n_test": len(test_idx),
        "n_features": len(features),
        "calibration_mode": calibration_mode,
    })

    fold_metrics = {
        "target": target,
        "protocol": protocol,
        "split_name": split_name,
        "fold_id": fold_id,
        "heldout_group": heldout_group,
        "model_name": "ExtraTrees_uncertainty",
        "n_train": len(train_idx),
        "n_fit": len(X_fit),
        "n_calibration": len(X_calib),
        "n_test": len(test_idx),
        "n_features": len(features),
        "calibration_mode": calibration_mode,
        "fit_predict_seconds": time.time() - t0,
        "rmse": rmse(y_test, y_test_pred),
        "mae": float(mean_absolute_error(y_test, y_test_pred)),
        "r2": float(r2_score(y_test, y_test_pred)) if len(y_test) >= 2 else np.nan,
        "spearman": spearman_safe(y_test, y_test_pred),
        "mean_tree_pred_std": float(np.nanmean(tree_std)),
        "median_tree_pred_std": float(np.nanmedian(tree_std)),
        "conformal_qhat": qhat,
        "conformal_coverage": float(np.mean(covered)),
        "median_conformal_width": float(np.nanmedian(conformal_width)),
        "mean_knn_distance": float(np.nanmean(test_knn_mean_distance)),
        "median_knn_distance": float(np.nanmedian(test_knn_mean_distance)),
        "pct_in_domain": float(np.mean(ad_flags == "in_domain")),
        "pct_borderline": float(np.mean(ad_flags == "borderline")),
        "pct_out_of_domain": float(np.mean(ad_flags == "out_of_domain")),
        "spearman_abs_error_vs_tree_std": spearman_safe(abs_error, tree_std),
        "spearman_abs_error_vs_knn_distance": spearman_safe(abs_error, test_knn_mean_distance),
        "spearman_abs_error_vs_domain_percentile": spearman_safe(abs_error, domain_pct),
    }

    return pred_rows, fold_metrics

print("Core uncertainty fold runner ready.")


In [ ]:
# ============================================================
# Cell 8: Run uncertainty and applicability-domain benchmark
# ============================================================

all_new_pred_chunks = []
all_new_fold_metrics = []
config_execution_rows = []

# Load existing checkpoint rows if present.
if PRED_CHECKPOINT.exists():
    print(f"Prediction checkpoint exists: {PRED_CHECKPOINT}")
if FOLD_CHECKPOINT.exists():
    print(f"Fold-metric checkpoint exists: {FOLD_CHECKPOINT}")

for _, cfg in config_plan_df.iterrows():
    config_key = cfg["config_key"]
    target = cfg["target"]
    protocol = cfg["protocol"]
    split_name = cfg["split_name"]

    if config_key in completed_ok:
        print(f"Skipping completed config: {config_key}")
        continue

    print("\n" + "=" * 90)
    print(f"Running config: {config_key}")
    print("=" * 90)

    cfg_t0 = time.time()
    try:
        df, X, y, features, mask_col = prepare_xy(target, protocol)
        splits = make_splits(df, split_name, seed=RANDOM_SEED)

        if not splits:
            raise ValueError(f"No valid splits generated for {config_key}")

        config_pred_chunks = []
        config_fold_metrics = []
        n_fold_errors = 0

        for fold_id, heldout_group, train_idx, test_idx in splits:
            if len(train_idx) < MIN_TRAIN_ROWS or len(test_idx) < MIN_TEST_ROWS:
                n_fold_errors += 1
                config_fold_metrics.append({
                    "target": target,
                    "protocol": protocol,
                    "split_name": split_name,
                    "fold_id": fold_id,
                    "heldout_group": heldout_group,
                    "status": "SKIPPED_INSUFFICIENT_ROWS",
                    "error": f"train={len(train_idx)}, test={len(test_idx)}",
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "n_features": len(features),
                })
                continue

            try:
                pred_rows, fold_metrics = fit_predict_uncertainty_one_fold(
                    df=df,
                    X=X,
                    y=y,
                    features=features,
                    train_idx=train_idx,
                    test_idx=test_idx,
                    target=target,
                    protocol=protocol,
                    split_name=split_name,
                    fold_id=fold_id,
                    heldout_group=heldout_group,
                )
                fold_metrics["status"] = "OK"
                fold_metrics["error"] = ""
                config_pred_chunks.append(pred_rows)
                config_fold_metrics.append(fold_metrics)
                print(f"  fold {fold_id:03d}: OK | test={len(test_idx)} | coverage={fold_metrics['conformal_coverage']:.3f} | OOD={fold_metrics['pct_out_of_domain']:.3f}")
            except Exception as fold_exc:
                n_fold_errors += 1
                err = traceback.format_exc()
                config_fold_metrics.append({
                    "target": target,
                    "protocol": protocol,
                    "split_name": split_name,
                    "fold_id": fold_id,
                    "heldout_group": heldout_group,
                    "status": "ERROR",
                    "error": str(fold_exc),
                    "traceback": err,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "n_features": len(features),
                })
                log_event("fold_run", "ERROR", "Fold failed.", {"config_key": config_key, "fold_id": fold_id, "error": str(fold_exc)})

        if config_pred_chunks:
            cfg_pred_df = pd.concat(config_pred_chunks, ignore_index=True)
            write_header = not PRED_CHECKPOINT.exists()
            cfg_pred_df.to_csv(PRED_CHECKPOINT, mode="a", header=write_header, index=False)
            all_new_pred_chunks.append(cfg_pred_df)

        cfg_fold_df = pd.DataFrame(config_fold_metrics)
        write_header = not FOLD_CHECKPOINT.exists()
        cfg_fold_df.to_csv(FOLD_CHECKPOINT, mode="a", header=write_header, index=False)
        all_new_fold_metrics.append(cfg_fold_df)

        ok_folds = int((cfg_fold_df["status"] == "OK").sum()) if "status" in cfg_fold_df.columns else 0
        status = "OK" if ok_folds > 0 and n_fold_errors == 0 else ("PARTIAL" if ok_folds > 0 else "ERROR")

        config_execution_rows.append({
            "config_key": config_key,
            "target": target,
            "protocol": protocol,
            "split_name": split_name,
            "status": status,
            "n_folds_generated": len(splits),
            "n_ok_folds": ok_folds,
            "n_fold_errors": n_fold_errors,
            "n_rows": len(df),
            "n_features": len(features),
            "mask_used": mask_col,
            "seconds": time.time() - cfg_t0,
            "error": "",
        })

    except Exception as cfg_exc:
        err = traceback.format_exc()
        config_execution_rows.append({
            "config_key": config_key,
            "target": target,
            "protocol": protocol,
            "split_name": split_name,
            "status": "ERROR",
            "n_folds_generated": 0,
            "n_ok_folds": 0,
            "n_fold_errors": 0,
            "n_rows": np.nan,
            "n_features": np.nan,
            "mask_used": "",
            "seconds": time.time() - cfg_t0,
            "error": str(cfg_exc),
            "traceback": err,
        })
        log_event("config_run", "ERROR", "Config failed.", {"config_key": config_key, "error": str(cfg_exc)})

    # Update config checkpoint after every config.
    new_config_df = pd.DataFrame(config_execution_rows)
    if CONFIG_CHECKPOINT.exists():
        old_config_df = pd.read_csv(CONFIG_CHECKPOINT)
        combined_config_df = pd.concat([old_config_df, new_config_df], ignore_index=True)
        combined_config_df = combined_config_df.drop_duplicates(subset=["config_key"], keep="last")
    else:
        combined_config_df = new_config_df.copy()
    combined_config_df.to_csv(CONFIG_CHECKPOINT, index=False)
    save_event_log()

print("\nUncertainty benchmark execution loop complete.")


In [ ]:
# ============================================================
# Cell 9: Consolidate checkpoint files into final processed outputs
# ============================================================

if not PRED_CHECKPOINT.exists():
    raise FileNotFoundError("Prediction checkpoint was not created. No successful uncertainty predictions found.")
if not FOLD_CHECKPOINT.exists():
    raise FileNotFoundError("Fold metrics checkpoint was not created.")
if not CONFIG_CHECKPOINT.exists():
    raise FileNotFoundError("Config execution checkpoint was not created.")

uncertainty_predictions_df = pd.read_csv(PRED_CHECKPOINT, low_memory=False)
uncertainty_fold_metrics_df = pd.read_csv(FOLD_CHECKPOINT, low_memory=False)
config_execution_df = pd.read_csv(CONFIG_CHECKPOINT, low_memory=False)

# Remove duplicate rows that may appear if a config was rerun after partial checkpoint.
pred_dedup_keys = [
    "target", "protocol", "split_name", "fold_id", "model_name", "record_index"
]
uncertainty_predictions_df = uncertainty_predictions_df.drop_duplicates(subset=pred_dedup_keys, keep="last")

fold_dedup_keys = ["target", "protocol", "split_name", "fold_id", "model_name"]
uncertainty_fold_metrics_df = uncertainty_fold_metrics_df.drop_duplicates(subset=fold_dedup_keys, keep="last")

# Save final processed outputs.
uncertainty_predictions_path = PROCESSED_DIR / "07_uncertainty_predictions.csv"
uncertainty_fold_metrics_path = PROCESSED_DIR / "07_uncertainty_fold_metrics.csv"
config_execution_path = AUDIT_DIR / "07_config_execution_audit.csv"

uncertainty_predictions_df.to_csv(uncertainty_predictions_path, index=False)
uncertainty_fold_metrics_df.to_csv(uncertainty_fold_metrics_path, index=False)
config_execution_df.to_csv(config_execution_path, index=False)

print(f"Saved: {uncertainty_predictions_path} | shape={uncertainty_predictions_df.shape}")
print(f"Saved: {uncertainty_fold_metrics_path} | shape={uncertainty_fold_metrics_df.shape}")
print(f"Saved: {config_execution_path} | shape={config_execution_df.shape}")

display(config_execution_df.groupby("status").size().reset_index(name="n_configs"))
display(uncertainty_fold_metrics_df.head())


In [ ]:
# ============================================================
# Cell 10: Aggregate uncertainty, coverage, and AD summaries
# ============================================================

pred = uncertainty_predictions_df.copy()
foldm = uncertainty_fold_metrics_df.copy()

# Ensure numeric columns.
for col in [
    "absolute_error", "tree_pred_std", "conformal_covered", "conformal_width",
    "knn_mean_distance", "domain_distance_percentile", "n_train", "n_test"
]:
    if col in pred.columns:
        pred[col] = pd.to_numeric(pred[col], errors="coerce")

# Group-level summary from predictions.
summary_group_cols = ["target", "protocol", "split_name"]
summary_rows = []
for keys, sub in pred.groupby(summary_group_cols, dropna=False):
    target, protocol, split_name = keys
    summary_rows.append({
        "target": target,
        "protocol": protocol,
        "split_name": split_name,
        "n_predictions": len(sub),
        "n_unique_records": sub["record_index"].nunique(),
        "mae_from_predictions": float(np.nanmean(sub["absolute_error"])),
        "rmse_from_predictions": float(np.sqrt(np.nanmean(sub["absolute_error"] ** 2))),
        "mean_tree_pred_std": float(np.nanmean(sub["tree_pred_std"])),
        "median_tree_pred_std": float(np.nanmedian(sub["tree_pred_std"])),
        "conformal_coverage": float(np.nanmean(sub["conformal_covered"])),
        "median_conformal_width": float(np.nanmedian(sub["conformal_width"])),
        "mean_knn_distance": float(np.nanmean(sub["knn_mean_distance"])),
        "median_knn_distance": float(np.nanmedian(sub["knn_mean_distance"])),
        "median_domain_distance_percentile": float(np.nanmedian(sub["domain_distance_percentile"])),
        "pct_in_domain": float(np.mean(sub["ad_flag"] == "in_domain")),
        "pct_borderline": float(np.mean(sub["ad_flag"] == "borderline")),
        "pct_out_of_domain": float(np.mean(sub["ad_flag"] == "out_of_domain")),
        "spearman_abs_error_vs_tree_std": spearman_safe(sub["absolute_error"].to_numpy(), sub["tree_pred_std"].to_numpy()),
        "spearman_abs_error_vs_knn_distance": spearman_safe(sub["absolute_error"].to_numpy(), sub["knn_mean_distance"].to_numpy()),
        "spearman_abs_error_vs_domain_percentile": spearman_safe(sub["absolute_error"].to_numpy(), sub["domain_distance_percentile"].to_numpy()),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PROCESSED_DIR / "07_calibration_and_domain_summary.csv", index=False)

# AD flag counts.
ad_flag_counts_df = (
    pred.groupby(["target", "protocol", "split_name", "ad_flag"], dropna=False)
    .size()
    .reset_index(name="n_predictions")
)
ad_flag_counts_df["pct_within_config"] = ad_flag_counts_df.groupby(["target", "protocol", "split_name"])["n_predictions"].transform(
    lambda x: 100.0 * x / x.sum()
)
ad_flag_counts_df.to_csv(AUDIT_DIR / "07_applicability_domain_flag_counts.csv", index=False)

# Conformal coverage by config.
coverage_df = summary_df[[
    "target", "protocol", "split_name", "n_predictions", "conformal_coverage", "median_conformal_width",
    "pct_in_domain", "pct_borderline", "pct_out_of_domain"
]].copy()
coverage_df["nominal_coverage"] = 1 - ALPHA
coverage_df["coverage_gap_vs_nominal"] = coverage_df["conformal_coverage"] - coverage_df["nominal_coverage"]
coverage_df.to_csv(AUDIT_DIR / "07_conformal_coverage_by_config.csv", index=False)

# Uncertainty-vs-error summary.
uncertainty_vs_error_df = summary_df[[
    "target", "protocol", "split_name", "n_predictions",
    "spearman_abs_error_vs_tree_std", "spearman_abs_error_vs_knn_distance", "spearman_abs_error_vs_domain_percentile",
    "mean_tree_pred_std", "median_tree_pred_std", "median_domain_distance_percentile"
]].copy()
uncertainty_vs_error_df.to_csv(AUDIT_DIR / "07_uncertainty_vs_error_summary.csv", index=False)

# Error by AD flag.
error_by_ad_flag_df = (
    pred.groupby(["target", "protocol", "split_name", "ad_flag"], dropna=False)
    .agg(
        n_predictions=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.nanmean(pd.to_numeric(x, errors="coerce"))))),
        median_tree_pred_std=("tree_pred_std", "median"),
        median_knn_distance=("knn_mean_distance", "median"),
        conformal_coverage=("conformal_covered", "mean"),
    )
    .reset_index()
)
error_by_ad_flag_df.to_csv(PROCESSED_DIR / "07_error_by_applicability_domain_flag.csv", index=False)

# Domain thresholds per config.
domain_threshold_rows = []
for keys, sub in pred.groupby(summary_group_cols, dropna=False):
    target, protocol, split_name = keys
    domain_threshold_rows.append({
        "target": target,
        "protocol": protocol,
        "split_name": split_name,
        "knn_distance_p50": float(np.nanpercentile(sub["knn_mean_distance"], 50)),
        "knn_distance_p90": float(np.nanpercentile(sub["knn_mean_distance"], 90)),
        "knn_distance_p97": float(np.nanpercentile(sub["knn_mean_distance"], 97)),
        "knn_distance_p99": float(np.nanpercentile(sub["knn_mean_distance"], 99)),
    })

domain_thresholds_df = pd.DataFrame(domain_threshold_rows)
domain_thresholds_df.to_csv(AUDIT_DIR / "07_domain_distance_thresholds.csv", index=False)

print("Saved uncertainty and AD summaries.")
display(summary_df.head(20))
display(error_by_ad_flag_df.head(20))


In [ ]:
# ============================================================
# Cell 11: Reliability bins for uncertainty and domain distance
# ============================================================

reliability_rows = []

for keys, sub in pred.groupby(["target", "protocol", "split_name"], dropna=False):
    target, protocol, split_name = keys
    sub = sub.copy()

    # Tree uncertainty bins.
    try:
        sub["tree_uncertainty_bin"] = pd.qcut(
            sub["tree_pred_std"], q=5, labels=["Q1_lowest", "Q2", "Q3", "Q4", "Q5_highest"], duplicates="drop"
        )
    except Exception:
        sub["tree_uncertainty_bin"] = "binning_failed"

    # Domain distance bins.
    try:
        sub["domain_distance_bin"] = pd.qcut(
            sub["knn_mean_distance"], q=5, labels=["Q1_nearest", "Q2", "Q3", "Q4", "Q5_farthest"], duplicates="drop"
        )
    except Exception:
        sub["domain_distance_bin"] = "binning_failed"

    for bin_col in ["tree_uncertainty_bin", "domain_distance_bin"]:
        for bin_name, b in sub.groupby(bin_col, dropna=False):
            reliability_rows.append({
                "target": target,
                "protocol": protocol,
                "split_name": split_name,
                "bin_type": bin_col,
                "bin": safe_str(bin_name),
                "n_predictions": len(b),
                "mae": float(np.nanmean(b["absolute_error"])),
                "median_absolute_error": float(np.nanmedian(b["absolute_error"])),
                "rmse": float(np.sqrt(np.nanmean(b["absolute_error"] ** 2))),
                "mean_tree_pred_std": float(np.nanmean(b["tree_pred_std"])),
                "median_knn_distance": float(np.nanmedian(b["knn_mean_distance"])),
                "conformal_coverage": float(np.nanmean(b["conformal_covered"])),
                "median_conformal_width": float(np.nanmedian(b["conformal_width"])),
            })

reliability_bins_df = pd.DataFrame(reliability_rows)
reliability_bins_df.to_csv(PROCESSED_DIR / "07_reliability_bins_uncertainty_and_domain.csv", index=False)

display(reliability_bins_df.head(30))
print("Saved reliability-bin diagnostics.")


In [ ]:
# ============================================================
# Cell 12: Compact reviewer-facing tables
# ============================================================

# Compact table emphasizing strict protocols and domain shift.
compact_cols = [
    "target", "protocol", "split_name", "n_predictions", "mae_from_predictions", "rmse_from_predictions",
    "conformal_coverage", "median_conformal_width", "pct_in_domain", "pct_borderline", "pct_out_of_domain",
    "spearman_abs_error_vs_tree_std", "spearman_abs_error_vs_knn_distance"
]
compact_uncertainty_table_df = summary_df[compact_cols].copy()
compact_uncertainty_table_df = compact_uncertainty_table_df.sort_values(["target", "protocol", "split_name"])
compact_uncertainty_table_df.to_csv(PROCESSED_DIR / "07_compact_uncertainty_ad_table.csv", index=False)

# Split-level summary focusing on leave-working-ion-out.
lwoo_table_df = compact_uncertainty_table_df[
    compact_uncertainty_table_df["split_name"] == "leave_working_ion_out"
].copy()
lwoo_table_df.to_csv(PROCESSED_DIR / "07_leave_working_ion_out_uncertainty_table.csv", index=False)

# OOD records table is not a ranking table; it is an audit of high-domain-distance predictions.
ood_audit_df = pred[pred["ad_flag"] == "out_of_domain"].copy()
if not ood_audit_df.empty:
    # Keep only audit columns and sort by domain percentile / error. This is diagnostic, not candidate ranking.
    ood_audit_df = ood_audit_df[[
        "target", "protocol", "split_name", "fold_id", "heldout_group", "record_index", "electrode_uid",
        "working_ion", "coarse_family", "host_chemsys_no_working_ion", "true_value", "predicted_value",
        "absolute_error", "tree_pred_std", "conformal_width", "knn_mean_distance", "domain_distance_percentile", "ad_flag"
    ]].sort_values(["target", "protocol", "split_name", "domain_distance_percentile", "absolute_error"], ascending=[True, True, True, False, False])

ood_audit_df.to_csv(AUDIT_DIR / "07_out_of_domain_prediction_audit.csv", index=False)

print("Compact tables saved:")
print(" - processed/07_compact_uncertainty_ad_table.csv")
print(" - processed/07_leave_working_ion_out_uncertainty_table.csv")
print(" - audit/07_out_of_domain_prediction_audit.csv")

display(compact_uncertainty_table_df.head(20))


In [ ]:
# ============================================================
# Cell 13: Final decision and output manifest
# ============================================================

def list_output_files(base_dir: Path):
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(base_dir)),
                "size_bytes": path.stat().st_size,
                "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    return pd.DataFrame(rows)

# Final audit values.
n_config_ok = int((config_execution_df["status"] == "OK").sum()) if "status" in config_execution_df.columns else 0
n_config_partial = int((config_execution_df["status"] == "PARTIAL").sum()) if "status" in config_execution_df.columns else 0
n_config_error = int((config_execution_df["status"] == "ERROR").sum()) if "status" in config_execution_df.columns else 0

n_fold_ok = int((uncertainty_fold_metrics_df["status"] == "OK").sum()) if "status" in uncertainty_fold_metrics_df.columns else 0
n_fold_error = int((uncertainty_fold_metrics_df["status"] == "ERROR").sum()) if "status" in uncertainty_fold_metrics_df.columns else 0
n_prediction_rows = len(uncertainty_predictions_df)

required_outputs = [
    PROCESSED_DIR / "07_uncertainty_predictions.csv",
    PROCESSED_DIR / "07_uncertainty_fold_metrics.csv",
    PROCESSED_DIR / "07_calibration_and_domain_summary.csv",
    PROCESSED_DIR / "07_error_by_applicability_domain_flag.csv",
    PROCESSED_DIR / "07_reliability_bins_uncertainty_and_domain.csv",
    PROCESSED_DIR / "07_compact_uncertainty_ad_table.csv",
    AUDIT_DIR / "07_config_execution_audit.csv",
    AUDIT_DIR / "07_applicability_domain_flag_counts.csv",
    AUDIT_DIR / "07_conformal_coverage_by_config.csv",
    AUDIT_DIR / "07_uncertainty_vs_error_summary.csv",
    AUDIT_DIR / "07_domain_distance_thresholds.csv",
    AUDIT_DIR / "07_out_of_domain_prediction_audit.csv",
]
missing_outputs = [str(p) for p in required_outputs if not p.exists()]

if missing_outputs:
    FINAL_DECISION_11 = "NO_GO_FIX_NOTEBOOK_07_OUTPUTS"
elif n_config_error > 0:
    FINAL_DECISION_11 = "CONDITIONAL_GO_REVIEW_CONFIG_ERRORS"
elif n_config_partial > 0:
    FINAL_DECISION_11 = "CONDITIONAL_GO_REVIEW_PARTIAL_CONFIGS"
elif n_fold_error > 0:
    FINAL_DECISION_11 = "CONDITIONAL_GO_REVIEW_FOLD_ERRORS"
elif n_prediction_rows == 0:
    FINAL_DECISION_11 = "NO_GO_NO_UNCERTAINTY_PREDICTIONS"
else:
    FINAL_DECISION_11 = "FULL_GO_TO_NOTEBOOK_12"

final_decision = {
    "final_decision": FINAL_DECISION_11,
    "n_config_ok": n_config_ok,
    "n_config_partial": n_config_partial,
    "n_config_error": n_config_error,
    "n_fold_ok": n_fold_ok,
    "n_fold_error": n_fold_error,
    "n_prediction_rows": n_prediction_rows,
    "missing_outputs": missing_outputs,
    "no_candidate_ranking": True,
    "no_cde_matching": True,
    "no_criticality_filtering": True,
    "no_manuscript_writing": True,
}

write_json_safe(final_decision, METADATA_DIR / "07_final_decision.json")

output_manifest_df = list_output_files(BASE_DIR)
output_manifest_df.to_csv(METADATA_DIR / "07_output_file_manifest.csv", index=False)

# Save final event log.
save_event_log()

print("\n" + "=" * 80)
print(f"Notebook 07 FINAL DECISION: {FINAL_DECISION_11}")
print("=" * 80)
print(json.dumps(final_decision, indent=2))

print("\nKey outputs:")
for p in required_outputs:
    print(f" - {p}  {'[OK]' if p.exists() else '[MISSING]'}")

display(output_manifest_df)
print("\nNotebook 11 complete.")
